# Chapter 5: Formatting Output & Structured Responses

**Technique:** Output formatting and structured responses

**Contents**
* [Lesson: Controlling Output Shape](#lesson)
* [Exercises](#exercises)
* [Example Playground](#example-playground)

In [ ]:
# Make the project root importable from any working directory, then load the helpers.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lib" / "helpers.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lib.helpers import get_completion, response_text, client, MODEL, MAX_TOKENS, SAMPLING_PARAMS

<a id="lesson"></a>
## Lesson: Controlling Output Shape

When you consume Claude's output in a program, not just display it to a human, you need the response to arrive in a predictable shape. Four techniques give you that control:

### 1. Explicit "no preamble" instructions

By default Claude may open with a sentence like "Sure, here's my review…". Tell it not to:

```python
PROMPT = "Review this code. Skip any preamble, output only the review.\nlast = items[len(items)]"
```

This works for any format: JSON, markdown tables, bullet lists. The instruction travels inside the user message, so no extra API parameters are needed.

### 2. XML tags for structured prose

Ask Claude to wrap sections in XML tags and parse (or highlight) them downstream:

```python
PROMPT = "Review this line and wrap your answer in <review></review> tags.\nlast = items[len(items)]"
```

Tags give you a reliable string boundary to extract with a regex (`re.search(r"<review>(.*?)</review>", text, re.S)`) or a simple `split`. They also let you request *multiple named sections* in one call, for example `<summary>` and `<risks>`.

### 3. Structured outputs via `output_config`

For machine readable payloads, use the `output_config` parameter with a JSON Schema. The API guarantees the response text is valid JSON that conforms to your schema:

```python
import json

msg = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[{"role": "user", "content": "Extract issues from this review note: 'off-by-one in last_item; also missing None check'."}],
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "summary": {"type": "string"},
                    "issues": {"type": "array", "items": {"type": "string"}},
                },
                "required": ["summary", "issues"],
                "additionalProperties": False,
            },
        },
    },
)
data = json.loads(response_text(msg))
```

Important details:
* Call `client.messages.create` directly (not the `get_completion` helper) so you can pass `output_config`.
* Always include `"additionalProperties": False`; without it the schema is underspecified and the model may add unexpected fields.
* The response is still a string; call `json.loads` to get a `dict`. Read it with `response_text(msg)` rather than `msg.content[0].text`: on current models the first content block can be a `thinking` block.

**Pydantic models.** The Python SDK can derive the schema from a Pydantic model and validate the result for you with `client.messages.parse(..., output_format=MyModel)`; the validated object is on `response.parsed_output`. There is an example in the cells below.

### 4. Stop sequences

Pass a `stop_sequences` list to truncate the response at a sentinel string. Useful when you want only the first section of a longer potential output:

```python
msg = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[{"role": "user", "content": "Review this line inside <review></review> tags, then suggest a fix: last = items[len(items)]"}],
    stop_sequences=["</review>"],
)
print(msg.stop_reason, msg.stop_sequence)  # "stop_sequence", "</review>"
```

### Historical note: assistant turn prefilling

Older Anthropic models accepted a partial `assistant` turn at the end of the messages list, letting you "speak for Claude" and force a specific opening. On current models (Sonnet 4.6 and later, including the course default `claude-opus-5-5`) that pattern returns a **400 Bad Request**, so the four techniques above replace it entirely.

In [ ]:
print(get_completion("Review this line and wrap your answer in <review></review> tags: last = items[len(items)]"))

In [ ]:
import json

msg = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[{"role": "user", "content": "Extract issues from this review note: 'off-by-one in last_item; also missing None check'."}],
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "summary": {"type": "string"},
                    "issues": {"type": "array", "items": {"type": "string"}},
                },
                "required": ["summary", "issues"],
                "additionalProperties": False,
            },
        },
    },
    **SAMPLING_PARAMS,
)
data = json.loads(response_text(msg))
print(type(data))
print(json.dumps(data, indent=2))

In [ ]:
# Pydantic: let a model class define the schema and validate the response.
from pydantic import BaseModel


class ReviewNote(BaseModel):
    summary: str
    issues: list[str]


parsed = client.messages.parse(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[{"role": "user", "content": "Extract issues from this review note: 'off-by-one in last_item; also missing None check'."}],
    output_format=ReviewNote,
    **SAMPLING_PARAMS,
)
note = parsed.parsed_output  # a validated ReviewNote instance
print(note.summary)
for issue in note.issues:
    print("-", issue)

In [ ]:
# Stop sequences: generation halts as soon as Claude emits the sentinel string.
msg = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[{"role": "user", "content": "Review this line. Put the review in <review></review> tags, then add a <fix></fix> section with corrected code: last = items[len(items)]"}],
    stop_sequences=["</review>"],
    **SAMPLING_PARAMS,
)
print(response_text(msg))
print("\nstop_reason:", msg.stop_reason, "| stop_sequence:", msg.stop_sequence)

<a id="exercises"></a>
## Exercises

### Exercise 5.1: Wrap a code review in `<review>` tags

**Scenario**: your CI pipeline parses Claude's code review output by looking for a `<review>` XML tag. The grader simulates that parser.

**Task**: write a prompt that asks Claude to review this line, `last = items[len(items)]`, and wrap its answer in `<review></review>` tags.

**Success criteria**: Claude's response contains the string `<review>`.

In [ ]:
from lib.grading import includes_any, grade

PROMPT = "[Replace this text]"

response = get_completion(PROMPT)

def grade_exercise(text):
    return includes_any(text, ["<review>"])

grade(response, grade_exercise(response))

In [ ]:
from hints import exercise_5_1_hint
print(exercise_5_1_hint)

### Exercise 5.2: Two delimited sections: `<summary>` and `<risks>`

**Scenario**: a security review tool expects two separate XML sections (a brief summary and a risk list) so each can be routed to a different downstream handler.

**Task**: write a prompt that asks Claude to review the code snippet `requests.get(user_input)` and return **two sections**: one wrapped in `<summary></summary>` tags and one wrapped in `<risks></risks>` tags.

**Success criteria**: Claude's response contains both `<summary>` and `<risks>`.

In [ ]:
from lib.grading import includes_all, grade

PROMPT = "[Replace this text]"

response = get_completion(PROMPT)

def grade_exercise(text):
    return includes_all(text, ["<summary>", "<risks>"])

grade(response, grade_exercise(response))

In [ ]:
from hints import exercise_5_2_hint
print(exercise_5_2_hint)

### Exercise 5.3: Structured output with `output_config`

**Scenario**: a code analysis pipeline needs machine readable output it can pass directly to a database insert, with no postprocessing regex and no fragile text parsing.

**Task**: fill in the `schema` dict so Claude's response parses to a Python `dict` with:
* `summary`, a `string` field
* `issues`, an `array` of strings

The `USER` message and the API call are already wired up; you only need to replace the `schema` placeholder. (While `schema` is still empty the API rejects the request; the cell catches that error and grades it as not yet solved.)

**Success criteria**: `json.loads(response_text(msg))` produces a dict where `isinstance(obj["summary"], str)` and `isinstance(obj["issues"], list)` are both true.

In [ ]:
import json

import anthropic
from lib.grading import grade

USER = "Summarize and list the problems in: 'no input validation; SQL string concatenation; no tests'."
schema = {
    # [Replace this dict so the output has: summary (string) and issues (array of strings)]
}

text, parsed = "", None
try:
    msg = client.messages.create(
        model=MODEL, max_tokens=MAX_TOKENS,
        messages=[{"role": "user", "content": USER}],
        output_config={"format": {"type": "json_schema", "schema": schema}},
        **SAMPLING_PARAMS,
    )
    text = response_text(msg)
    parsed = json.loads(text)
except (anthropic.BadRequestError, json.JSONDecodeError) as err:
    text = text or f"Request failed: {err}"

def grade_exercise(obj):
    return isinstance(obj, dict) and isinstance(obj.get("summary"), str) and isinstance(obj.get("issues"), list)

grade(text, grade_exercise(parsed))

In [ ]:
from hints import exercise_5_3_hint
print(exercise_5_3_hint)

## Common mistakes, stretch challenge, and reflection

**Common mistakes**

* **Trying to prefill the assistant turn**: passing `{"role": "assistant", "content": "..."}` as the last message returns a **400 Bad Request** on current models. Use explicit instructions or `output_config` instead.
* **Forgetting `"additionalProperties": False`**: without it the JSON Schema is underspecified. Claude may add extra fields that break downstream code expecting exactly the fields you declared.
* **Not calling `json.loads`**: the response text is always a `str`. Even when structured output is enabled, you must parse it yourself before accessing keys (or use `client.messages.parse` with a Pydantic model).
* **Reading `msg.content[0].text`**: on current models `content[0]` may be a `thinking` block with no `.text`. Select the text blocks (`response_text(msg)`).

**Stretch challenge**

Extend the Exercise 5.3 schema to include a `severity` field with an `enum` constraint (`["low", "medium", "high"]`). Add `"severity"` to the `required` list, rerun, and observe how Claude populates it. Then do the same with a Pydantic model using `Literal["low", "medium", "high"]`.

**Reflect**: when is structured output (`output_config`) better than asking Claude to return JSON in free text and parsing it with a regex? Consider error handling, schema evolution, and the cost of a malformed response in a production pipeline.

<a id="example-playground"></a>
## Example Playground

Use the cells below to experiment freely. Try combining XML tags with `output_config`, add stop sequences, or request additional schema fields.

In [ ]:
# Experiment: structured output with an extra "severity" field
import json

schema = {
    "type": "object",
    "properties": {
        "summary": {"type": "string"},
        "issues": {"type": "array", "items": {"type": "string"}},
        "severity": {"type": "string", "enum": ["low", "medium", "high"]},
    },
    "required": ["summary", "issues", "severity"],
    "additionalProperties": False,
}

msg = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[{"role": "user", "content": 'Analyze this Flask code: return requests.get(request.args["url"]).text'}],
    output_config={"format": {"type": "json_schema", "schema": schema}},
    **SAMPLING_PARAMS,
)
print(json.dumps(json.loads(response_text(msg)), indent=2))